# 14 · Taxonomy Analysis
**Explore and compare taxonomy versions; find coverage gaps and unused codes.**

| Step | What happens |
|------|--------------|
| 1 | Load and visualise the current taxonomy structure |
| 2 | Diff two taxonomy versions using `scripts/taxonomy_diff.py` |
| 3 | Code distribution from a classification run |
| 4 | Coverage gaps — codes never assigned in the sample |
| 5 | Quality checks — missing labels / descriptions |

In [ ]:
WORKSPACE = "scopus_content"  # folder under use_cases/ to work in — change to run on another use case
# ── Project root detection ────────────────────────────────────────────────
import sys, os
from pathlib import Path

_root = Path().absolute()
while not (_root / "use_cases").is_dir() and _root.parent != _root:
    _root = _root.parent
_root = _root / "use_cases" / WORKSPACE  # the use-case folder this notebook works in
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f"Project root: {os.getcwd()}")

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

TAXONOMY_DIR   = "generated/taxonomies"
PRIOR_RUN      = "runs/output/tour_sample_enriched.xlsx"   # notebook 10 output
MODULE         = "contact_reason"

## Step 1 · Visualise the current taxonomy

In [ ]:
from rca_pipeline.taxonomy import load_taxonomies, leaf_label_lookup, leaf_description_lookup

taxonomies = load_taxonomies(TAXONOMY_DIR)
tax = taxonomies.get(MODULE, {})

print(f"Module: {MODULE}")
print(f"L1 nodes : {len([v for v in tax.values() if isinstance(v, dict)])}")
l2_count = sum(len(v.get('children', {})) for v in tax.values() if isinstance(v, dict))
print(f"L2 leaves: {l2_count}")
print()

# Tree view
for l1_code, l1_node in sorted(tax.items()):
    if not isinstance(l1_node, dict):
        continue
    children = l1_node.get("children", {})
    print(f"  {l1_code}  {l1_node.get('label', '')}  ({len(children)} children)")

In [ ]:
# Bar chart: number of L2 codes per L1 group
l1_labels = []
l2_counts = []

for l1_code, l1_node in sorted(tax.items()):
    if not isinstance(l1_node, dict):
        continue
    children = l1_node.get("children", {})
    label = l1_node.get("label", l1_code)
    l1_labels.append(f"{l1_code}\n{label[:20]}")
    l2_counts.append(len(children))

if l1_labels:
    fig, ax = plt.subplots(figsize=(max(8, len(l1_labels) * 0.9), 4))
    ax.bar(l1_labels, l2_counts, color="steelblue")
    ax.set_ylabel("L2 leaf count")
    ax.set_title(f"{MODULE}: L2 codes per L1 group")
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    plt.xticks(fontsize=8)
    plt.tight_layout()
    plt.show()

## Step 2 · Diff two taxonomy versions
Compares any two taxonomy JSON files. Simulates an "old" version by stripping descriptions.

In [ ]:
import copy, tempfile, subprocess

# Find the contact_reason taxonomy file
tax_files = list(Path(TAXONOMY_DIR).glob("*contact_reason*.json"))
if not tax_files:
    tax_files = list(Path(TAXONOMY_DIR).glob("*.json"))

if tax_files:
    new_tax_path = tax_files[0]
    new_tax      = json.loads(new_tax_path.read_text(encoding="utf-8"))

    # Synthesise an "old" version: drop descriptions from all nodes
    old_tax = copy.deepcopy(new_tax)
    for node in old_tax.values():
        if isinstance(node, dict):
            node.pop("description", None)
            for child in node.get("children", {}).values():
                if isinstance(child, dict):
                    child.pop("description", None)

    # Write old version to a temp file
    with tempfile.NamedTemporaryFile(suffix=".json", mode="w",
                                      encoding="utf-8", delete=False) as tmp:
        json.dump(old_tax, tmp)
        tmp_path = tmp.name

    result = subprocess.run(
        [sys.executable, "scripts/taxonomy_diff.py", tmp_path, str(new_tax_path)],
        capture_output=True, text=True, encoding="utf-8"
    )
    print(result.stdout or result.stderr)
    Path(tmp_path).unlink(missing_ok=True)
else:
    print(f"No taxonomy JSON files found in {TAXONOMY_DIR}. Run notebook 11 first.")

## Step 3 · Code distribution from a classification run

In [ ]:
prior_path = Path(PRIOR_RUN)

if prior_path.exists():
    df_run = pd.read_excel(prior_path)
    code_col  = "contact_reason_primary_code"
    label_col = "contact_reason_primary_label"

    if code_col in df_run.columns:
        dist = df_run[label_col if label_col in df_run.columns else code_col].value_counts()
        print(f"Code distribution ({len(df_run)} rows):")
        print(dist.to_string())
    else:
        print(f"Column {code_col} not found. Columns: {list(df_run.columns)}")
else:
    print(f"No prior run at {PRIOR_RUN}. Run notebook 10 first.")
    df_run = None

In [ ]:
if df_run is not None and code_col in df_run.columns:
    dist_plot = (
        df_run[label_col if label_col in df_run.columns else code_col]
        .value_counts()
        .head(15)
        .sort_values()
    )
    fig, ax = plt.subplots(figsize=(8, max(4, len(dist_plot) * 0.35)))
    dist_plot.plot.barh(ax=ax, color="steelblue")
    ax.set_xlabel("Count")
    ax.set_title(f"{MODULE}: assigned code distribution (top 15)")
    plt.tight_layout()
    plt.show()

## Step 4 · Coverage gaps — codes never assigned

In [ ]:
# All codes defined in the taxonomy
all_codes = set()
for l1_code, l1_node in tax.items():
    if isinstance(l1_node, dict):
        all_codes.add(l1_code)
        for l2_code in l1_node.get("children", {}):
            all_codes.add(l2_code)

print(f"Total defined codes: {len(all_codes)}")

if df_run is not None and code_col in df_run.columns:
    used_codes = set(df_run[code_col].dropna().astype(str).unique())

    # Also check secondary codes if present
    alt_col = code_col.replace("_primary_code", "_secondary_code")
    if alt_col in df_run.columns:
        used_codes |= set(df_run[alt_col].dropna().astype(str).unique())

    unused = sorted(all_codes - used_codes)
    print(f"Codes used in sample : {len(used_codes)}")
    print(f"Codes NEVER assigned : {len(unused)}")
    if unused:
        print()
        label_lkp = leaf_label_lookup(taxonomies)
        for code in unused:
            lbl = label_lkp.get(code, "(no label)")
            print(f"  {code:12s}  {lbl}")
else:
    print("No run data available for coverage analysis.")

## Step 5 · Quality checks on taxonomy files

In [ ]:
issues = []

for mod_name, mod_tax in taxonomies.items():
    for l1_code, l1_node in mod_tax.items():
        if not isinstance(l1_node, dict):
            continue
        if not l1_node.get("label"):
            issues.append({"module": mod_name, "code": l1_code, "issue": "missing L1 label"})
        if not l1_node.get("description"):
            issues.append({"module": mod_name, "code": l1_code, "issue": "missing L1 description"})
        for l2_code, l2_node in l1_node.get("children", {}).items():
            if not isinstance(l2_node, dict):
                continue
            if not l2_node.get("label"):
                issues.append({"module": mod_name, "code": l2_code, "issue": "missing L2 label"})
            if not l2_node.get("description"):
                issues.append({"module": mod_name, "code": l2_code, "issue": "missing L2 description"})

if issues:
    df_issues = pd.DataFrame(issues)
    print(f"Quality issues found: {len(issues)}")
    print(df_issues.groupby(["module", "issue"]).size().rename("count").to_string())
    print()
    display(df_issues.head(20))
else:
    print("No quality issues found — all nodes have labels and descriptions.")